# Suite COL — Collections

A collection is a list (mbse-schemas' `OfIndexed`, positional or keyed) or an object's entries in an adjacency. Basic
reads a list property as a `Domains.Collection` and an object's entries as records, counts, indexes, tests membership,
adds, finds the least and the greatest and tests uniqueness, and binds a name to each item with the quantifiers `all`,
`any` and `count`, all by Kleene's logic. Collections have domains for inference only: `Domains.List` and
`Domains.Keyed`.

In [ ]:
from mbse.Expressions import Domains as D, Evaluators as V, Expressions as E, Translators as T
from mbse.Expressions.Dialects.Latex import Expressions as L
from mbse.Expressions.Framework import Domains as FD, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S
from support import raises, text
from mbse.Expressions import register

store = register(Proxies.OfStore())  # holds every dialect's meta-schemas


def listed(spec):
    return lambda t: t.as_indexed(lambda i: i.of(spec))


Int, Float, Str = (S.OfNative.Data(t) for t in (int, float, str))
Port = S.OfObject.Builder().name("Port").ref().properties(text("name"), text("width", int)).create()
Wire = S.OfRelation.Builder().name("Wire").links("component", "port").properties(text("pin", int)).create()
Component = S.OfObject.Builder().name("Component").ref().properties(
    text("name"), lambda p: p.name("sizes").of(listed(Int)), lambda p: p.name("grid").of(listed(listed(Int))),
    lambda p: p.name("limits").of(lambda t: t.as_indexed(lambda i: i.key(Str).of(Float))),
    lambda p: p.name("labels").of(listed(Str))).relations(lambda a: a.name("ports").of(Wire).me("component")).create()
for schema in [Port, Wire, Component]:
    store.register(schema)
B = store
a, b = B.Port().name("a").width(8).create(), B.Port().name("b").width(16).create()
amp = B.Component().name("amp").sizes([3, 1, 2]).grid([[1, 2], [3]]).limits({"lo": 0.5, "hi": 2.0}).labels(
    ["x", "y", "x"]).ports(lambda w: w.port(a).pin(1)).ports(lambda w: w.port(b).pin(2)).create()
this, x, xs = E.variable("this"), E.variable("x"), E.variable("xs")


def ev(expression, **variables):
    """The value of a Basic expression, with `this` bound to `amp` unless given."""
    return V.OfAny(expression, {"this": amp, **variables})

## COL-01 · A list property reads as a collection, positional or keyed, nested lists too; a scope's list is positional

In [ ]:
sizes = ev(this.sizes)
assert sizes == D.Collection((3, 1, 2)) and sizes.keys is None and repr(sizes) == "[3, 1, 2]"
assert ev(this.grid) == D.Collection((D.Collection((1, 2)), D.Collection((3,))))
limits = ev(this.limits)
assert limits == D.Collection((0.5, 2.0), ("lo", "hi")) and repr(limits) == "{'lo': 0.5, 'hi': 2.0}"
assert ev(this.get("missing")) is None and ev(this.has("sizes")) is True
assert ev(xs.count(), xs=[1, 2]) == 2 and ev(xs.count(), xs=(1, 2, 3)) == 3 and ev(xs.count(), xs=D.Collection(())) == 0
I8 = D.OfInteger.Data(8)
assert D.Collection((b"ab", D.Value(I8, 1))) == D.Collection((b"ab", D.Value(D.OfInteger.Data(8), 1)))
assert D.Collection((b"ab",)) != D.Collection((b"ac",)) and D.Collection((1,), ("k",)) != D.Collection((1,))
# Collections compare deeply: positional item by item, keyed by key in any order, records by field.
ys = E.variable("ys")
assert ev(this.sizes.eq(this.sizes)) is True and ev(this.grid.eq(this.grid)) is True and ev(this.limits.eq(this.limits)) is True
assert ev(xs.eq(ys), xs=[1, 2], ys=[1, 3]) is False and ev(xs.eq(ys), xs=[1, 2], ys=[1]) is False  # an item, a length
assert ev(xs.eq(ys), xs=[2, None], ys=[1, 2]) is False and ev(xs.eq(ys), xs=[1, None], ys=[1, 2]) is None  # Kleene's and
keyed = D.Collection((1, 2), ("a", "b"))
assert ev(xs.eq(ys), xs=keyed, ys=D.Collection((2, 1), ("b", "a"))) is True  # in any order
assert ev(xs.eq(ys), xs=keyed, ys=D.Collection((1, 3), ("a", "b"))) is False
assert ev(xs.eq(ys), xs=keyed, ys=D.Collection((1, 2), ("a", "c"))) is None  # other keys: incomparable
assert ev(xs.eq(ys), xs=keyed, ys=D.Collection((1,), ("a",))) is None and ev(xs.eq(ys), xs=keyed, ys=[1, 2]) is None
assert ev(xs.eq(ys), xs=D.Record({"n": 1}), ys=D.Record({"n": 1})) is True and ev(xs.eq(ys), xs=D.Record({"n": 1}), ys=D.Record({"m": 1})) is None
assert ev(xs.eq(ys), xs=[1], ys=D.Record({"n": 1})) is None and ev(xs.in_(ys), xs=[1], ys=[[2], [1]]) is True

## COL-02 · count, item and in: positions from 0 and keys, unknown where there is no item

In [ ]:
assert ev(this.sizes.count()) == 3 and ev(this.limits.count()) == 2
assert ev(this.sizes.item(0)) == 3 and ev(this.sizes.item(2)) == 2 and ev(this.sizes.item(3)) is None
assert ev(this.sizes.item(-1)) is None and ev(this.sizes.item(E.literal(1, D.OfInteger.Data(8)))) == 1
assert ev(this.limits.item("hi")) == 2.0 and ev(this.limits.item("mid")) is None and ev(this.grid.item(1).item(0)) == 3
assert ev(E.literal(2).in_(this.sizes)) is True and ev(E.literal(5).in_(this.sizes)) is False
assert ev(E.literal(2.0).in_(this.limits)) is True and ev(E.literal(1.0).in_(this.limits)) is False
assert ev(E.literal("lo").in_(this.limits)) is None  # items, not keys: a str is incomparable with them
assert ev(E.literal(2.0).in_(this.sizes)) is None  # incomparable with every item: unknown
assert ev(E.literal(1).in_(xs), xs=[1.0, 1]) is True and ev(E.literal(2).in_(xs), xs=[1.0, 1]) is None
for unknown in [this.get("nope").count(), this.get("nope").item(0), this.sizes.item(this.get("nope")),
                E.literal(1).in_(this.get("nope")), this.get("nope").in_(this.sizes)]:
    assert ev(unknown) is None
with raises(TypeError, match="count expects a collection, got int"):
    ev(E.literal(1).count())
with raises(TypeError, match="item expects an integer position, got str"):
    ev(this.sizes.item("0"))

## COL-03 · sum, min, max and unique, by the domains of the items

In [ ]:
assert ev(this.sizes.sum()) == 6 and ev(this.limits.sum()) == 2.5 and ev(xs.sum(), xs=[]) == 0
I8 = D.OfInteger.Data(8)
assert ev(xs.sum(), xs=[D.Value(I8, 100), D.Value(I8, 27)]) == D.Value(I8, 127)
with raises(OverflowError, match="sum overflows int8: 128"):
    ev(xs.sum(), xs=[D.Value(I8, 100), D.Value(I8, 28)])
with raises(TypeError, match="sum expects numbers of one domain, got int and float"):
    ev(xs.sum(), xs=[1, 2.0])
with raises(TypeError, match="sum expects numbers, got str"):
    ev(this.labels.sum())
assert ev(this.sizes.min()) == 1 and ev(this.sizes.max()) == 3 and ev(this.labels.max()) == "y"
assert ev(xs.min(), xs=[]) is None and ev(xs.max(), xs=[1.0, float("nan")]) is None  # incomparable
assert ev(xs.min(), xs=[D.Value(I8, 5), D.Value(I8, -5)]) == D.Value(I8, -5)
with raises(TypeError, match="min expects ordered values of one domain, got int and str"):
    ev(xs.min(), xs=[1, "a"])
with raises(TypeError, match="max expects ordered values of one domain, got bool and bool"):
    ev(xs.max(), xs=[True, False])
assert ev(this.sizes.unique()) is True and ev(this.labels.unique()) is False and ev(xs.unique(), xs=[1, 1.0]) is None
assert ev(xs.unique(), xs=[]) is True
for unknown in [this.get("nope").sum(), this.get("nope").min(), this.get("nope").unique()]:
    assert ev(unknown) is None
assert ev(xs.sum(), xs=[1, None]) is None and ev(xs.max(), xs=[1, None]) is None

## COL-04 · entries gives an object's entries in an adjacency as records, which get and has read

In [ ]:
ports = ev(this.entries("ports"))
assert len(ports.items) == 2 and ports.keys is None
first = ports.items[0]
assert isinstance(first, D.Record) and first.fields == {"port": a, "pin": 1} and repr(first).startswith("record(port=")
assert ev(this.entries("ports").item(1).get("pin")) == 2 and ev(this.entries("ports").item(0).has("pin")) is True
assert ev(this.entries("ports").item(0).get("port").eq(E.variable("a")), a=a) is True  # the linked object itself
assert ev(this.entries("ports").item(0).get("port").get("width")) == 8 and ev(this.entries("nope").count()) == 0
assert ev(E.variable("o").entries("ports"), o=None) is None
with raises(TypeError, match="entries expects an adjacency name, got int"):
    ev(this.entries(1))
with raises(TypeError, match="entries expects an object, got int"):
    ev(E.literal(1).entries("ports"))
with raises(TypeError, match="get expects an object, got Collection"):
    ev(this.sizes.get("x"))

## COL-05 · Quantifiers bind a name to each item: all and any by Kleene's logic, count of the items the body holds for

In [ ]:
assert ev(this.sizes.all("x", x.gt(0))) is True and ev(this.sizes.all("x", x.gt(1))) is False
assert ev(this.sizes.any("x", x.gt(2))) is True and ev(this.sizes.any("x", x.gt(5))) is False
assert ev(this.sizes.count_where("x", x.gt(1))) == 2 and ev(this.limits.all("x", x.gt(0.0))) is True
assert ev(xs.all("x", x.gt(0)), xs=[]) is True and ev(xs.any("x", x.gt(0)), xs=[]) is False and ev(xs.count_where("x", True), xs=[]) == 0
# Unknown bodies: false or true decide, otherwise unknown; count is unknown.
assert ev(xs.all("x", x.gt(0)), xs=[1, "a"]) is None and ev(xs.all("x", x.gt(0)), xs=["a", 0]) is False
assert ev(xs.any("x", x.gt(0)), xs=["a", 0]) is None and ev(xs.any("x", x.gt(0)), xs=["a", 1]) is True
assert ev(xs.count_where("x", x.gt(0)), xs=[1, "a"]) is None
assert ev(this.get("nope").all("x", True)) is None
# Over entries: every port's width is at least 8, and the pins are unique.
assert ev(this.entries("ports").all("e", E.variable("e").get("port").get("width").ge(8))) is True
assert ev(this.entries("ports").count_where("e", E.variable("e").get("pin").gt(1))) == 1
# Nested, and the name shadows an outer binding within the body only.
assert ev(this.grid.all("row", E.variable("row").any("x", x.ge(3)))) is False
assert ev(E.let_("x", 10, this.sizes.all("x", x.lt(10)).and_(x.eq(10)))) is True
with raises(TypeError, match="all expects bool operands, got int"):
    ev(this.sizes.all("x", x))
with raises(TypeError, match="any expects a collection, got int"):
    ev(E.literal(1).any("x", True))
assert V.OfQuantifier(E.quantifier("all", "x", xs, x.ge(1)), {"xs": [1, 2]}) is True

## COL-06 · The quantifier kind: built, validated as a binding, stored and read back, and translated nowhere yet

In [ ]:
built = E.OfQuantifier.Builder().name("x").quantifier("any").collection(xs).body(x.gt(1)).create()
assert built.form() == F.Form("quantifier", {"name": "x", "quantifier": "any"}, (xs.data, built.body))
assert F.same(E.DIALECT.make(built.form()), built) and F.same(E.OfAny.resolve(lambda b: b.as_quantifier(built)), built)
assert built.validate(bound={"xs"}) == [] and SY.free(built) == {"xs"}
assert built.validate() == ["collection: variable 'xs' is not bound"]
assert E.quantifier("all", "x", xs, E.variable("y")).data.validate(bound={"xs"}) == ["body: variable 'y' is not bound"]
assert E.quantifier("every", "x", xs, True).data.validate(bound={"xs"}, core=True) == ["'every' is not a core operation"]
assert E.OfQuantifier.Data("x", "all", xs.data).validate(bound={"xs"}) == ["a quantifier needs a body"]
assert E.OfQuantifier.Data(None, "all", xs.data, E.literal(True).data).validate(bound={"xs"}) == ["a quantifier needs a name"]
with raises(ValueError, match="a quantifier needs a body"):
    V.OfAny(E.OfQuantifier.Data("x", "all", xs.data), {"xs": []})
with raises(NotImplementedError, match="'every' is not a core operation"):
    V.OfAny(E.quantifier("every", "x", xs, True), {"xs": []})
text_ = JSON.ToJSON(store).Reachable(E.OfQuantifier.Schema, built)
back = JSON.FromJSON(E.Builders).Reachable(E.OfQuantifier.Schema, text_)
assert F.same(back, built) and E.Builders.name_of(E.OfQuantifier.Schema) == "Expressions.OfQuantifier"
with raises(ValueError, match="Basic quantifier 'all' has no Latex counterpart"):
    T.between(E.DIALECT, L.DIALECT).forward(this.sizes.all("x", x.gt(0)))

## COL-07 · Collections have domains for inference: a quantifier's name has the items' domain

In [ ]:
ints, keyed = D.List(D.Int), D.Keyed(D.Str, D.Float)
assert ints.name() == repr(ints) == "list(int)" and repr(keyed) == "keyed(str, float)" and ints == D.List(D.Int) and ints != D.List(D.Str)
assert D.List(D.OfInteger.Data(8)) == D.List(D.OfInteger.Data(8))
assert ints.contains(D.Collection((1, 2))) and ints.contains([1]) and not ints.contains(D.Collection((1,), ("a",)))
assert not ints.contains([1.5]) and keyed.contains(D.Collection((1.5,), ("a",))) and not keyed.contains(D.Collection((1.5,)))
assert ints.includes(D.List(D.Int)) and not ints.includes(D.List(D.Str)) and ints.includes(FD.OfUnion(ints))
assert keyed.includes(D.Keyed(D.Str, D.Float)) and keyed.includes(FD.OfUnion(keyed)) and not keyed.includes(ints)
assert D.items_of(ints) is D.Int and D.items_of(keyed) is D.Float and D.items_of(D.Int) is FD.Anything
assert D.Records.contains(D.Record({})) and not D.Records.contains(amp)
infer = E.DIALECT.infer
assert infer(xs.all("x", x.gt(0)), {"xs": ints}) is D.Bool and infer(xs.count_where("x", x.gt(0)), {"xs": ints}) is D.Int
assert infer(xs.sum(), {"xs": D.List(D.Float)}) is D.Float and infer(xs.item(0), {"xs": keyed}) is D.Float
assert infer(xs.min(), {"xs": D.List(D.Str)}) is D.Str and infer(xs.count(), {"xs": FD.Anything}) is D.Int
assert infer(E.literal(1).in_(xs), {"xs": ints}) is D.Bool and infer(xs.unique(), {"xs": ints}) is D.Bool
assert infer(this.entries("ports"), {"this": D.Object}) == D.List(D.Records)
assert infer(this.entries("ports").all("e", E.variable("e").get("pin").gt(1)), {"this": D.Object}) is D.Bool
assert infer(E.quantifier("every", "x", xs, x), {"xs": ints}) is FD.Anything  # an extension: its name is unknown too
with raises(TypeError, match="sum cannot take (list(str)); it takes (collection of number) -> number"):
    infer(xs.sum(), {"xs": D.List(D.Str)})
with raises(TypeError, match="count cannot take (int); it takes (collection of T) -> int"):
    infer(E.literal(1).count())
with raises(TypeError, match="in cannot take (int, int); it takes (T, collection of T) -> bool"):
    infer(E.literal(1).in_(2))
with raises(TypeError, match="all cannot take (list(int), int); it takes (collection, bool) -> bool"):
    infer(xs.all("x", x), {"xs": ints})
with raises(TypeError, match="any cannot take (int, bool); it takes (collection, bool) -> bool"):
    infer(E.literal(1).any("x", True))